# WEEK 7 - HOMEWORK

In [ ]:
from code_for_hw7 import *
import numpy as np
import modules_disp as disp

A code and data folder that will be useful for doing this lab can be found [**here**](https://introml_oll.odl.mit.edu/cat-soop/_static/6.036/homework/hw07/code_for_hw7.zip). Download this to your computer, or alternatively, use the [**Colab notebook**](https://colab.research.google.com/drive/14pGqETHUQpCCVlkAU4rB_ueh5RjzcqzC).

This homework continues the exploration and implementation of [**neural networks**](https://openlearninglibrary.mit.edu/courses/course-v1:MITx+6.036+1T2019/courseware/Week6/neural_networks/1) as discussed in the notes.

In particular, this homework considers neural networks with multiple layers. Each layer has multiple inputs and outputs, and can be broken down into two parts:

- A **linear** module that implements a linear transformation:

  $$
  z_j = \left(\sum_{i=1}^{m} x_i W_{i,j}\right) + W_{0,j}
  $$

  specified by a weight matrix $W$ and a bias vector $W_0$. The output is:

  $$
  [z_1,\ldots,z_n]^T
  $$

- An **activation** module that applies an activation function to the outputs of the linear module for some activation function $f$, such as **Tanh** or **ReLU** in the hidden layers, or **Softmax** at the output layer. We write the output as:

  $$
  [f(z_1),\ldots,f(z_m)]^T
  $$

  although technically, for some activation functions such as softmax, each output will depend on all the $z_i$, not just one.

We will use the following notation for quantities in a network:

- Inputs to the network are $x_1,\ldots,x_d$.
- Number of layers is $L$.
- There are $m_l$ inputs to layer $l$.
- There are $n_l = m_{l+1}$ outputs from layer $l$.
- The weight matrix for layer $l$ is $W^l$, an $m_l \times n_l$ matrix, and the bias vector (offset) is $W_0^l$, an $n_l \times 1$ vector.
- The outputs of the linear module for layer $l$ are known as **pre-activation** values and denoted $z^l$.
- The activation function at layer $l$ is $f^l(\cdot)$.
- Layer $l$ activations are:

  $$
  a^l =
  [f^l(z_1^l),\ldots,f^l(z_{n_l}^l)]^T
  $$

- The output of the network is:

  $$
  a^L =
  [f^L(z_1^L),\ldots,f^L(z_{n_L}^L)]^T
  $$

- The loss function $\operatorname{Loss}(a,y)$ measures the loss of output values $a$ when the target is $y$.

Here is an illustrative picture:

<div align='center'>
    <img src='../assets/2_network.png' alt='network_img' style = 'border-radius: 10px'/>
</div>

## 1) Backpropagation

The materials for week 6 and week 7 will be helpful here, including the [week6 lecture](https://openlearninglibrary.mit.edu/courses/course-v1\:MITx+6.036+1T2019/courseware/Week6/week6_video/1) and [week7 lecture](https://openlearninglibrary.mit.edu/courses/course-v1\:MITx+6.036+1T2019/courseware/Week7/week7_video/1).

We have seen in the [lecture notes](https://openlearninglibrary.mit.edu/courses/course-v1\:MITx+6.036+1T2019/courseware/Week6/neural_networks/5) how to train multi-layer neural networks as classifiers using stochastic gradient descent (SGD). One of the key steps in the SGD method is the evaluation of the gradient of the loss function with respect to the model parameters. In this problem, you will derive the backpropagation method for a general `L`-layer neural network. We'll exploit the decomposition of the network into *linear* and *activation* modules that we introduced at the start of this homework. Remember that we've defined the shapes of the various quantities at the start of the homework.

- Each linear module has a `forward` method that takes in a column vector of activations `A` (from the previous layer) and returns a column vector `Z` of pre-activations; it can also store its input or output vectors for use by other methods (e.g., for subsequent backpropagation).
- Each activation module has a `forward` method that takes in a column vector of pre-activations `Z` and returns a column vector `A` of activations; it can also store its input or output vectors for use by other methods (e.g., for subsequent backpropagation).
- Each linear module has a `backward` method that takes in a column vector of $\frac{\partial Loss}{\partial Z}$ and returns a column vector of $\frac{\partial Loss}{\partial A}$. This module also computes and stores $\frac{\partial Loss}{\partial W}$ and $\frac{\partial Loss}{\partial W_0}$, the gradients with respect to the weights.
- Each activation module has a `backward` method that takes in a column vector of $\frac{\partial Loss}{\partial A}$ and returns a column vector of $\frac{\partial Loss}{\partial Z}$.

The backpropagation algorithm will consist of:

- Calling the `forward` method of each module in turn, feeding the output of one module as the input to the next; starting with the input values of the network. After this pass, we have a predicted value for the final network output.
- Calling the `backward` method of each module in reverse order, using the returned value from one module as the input value of the previous one. The starting value for the backward method is $\frac{\partial Loss(a^L,y)}{\partial a^L}$, where $a^L$ is the activation of the final layer (computed during the forward pass) and $y$ is the desired output (the label).

### 1.1) Linear Module

The `forward` method, given `A` from the previous layer, implements:

$$
Z=W^TA+W_0
$$

and stores the input `A` to be used by the `backward` method.

Recall that there are `n_l=m_{l+1}` outputs from layer `l`. For layer `l`, `W` is a `m_l\times n_l` matrix, `W_0` is a `n_l\times1` vector, and `A` from the previous layer is a `n_{l-1}\times1` (or `m_l\times1`) vector. Given these shapes, make sure that you understand why the forward equation has `W^T` and not `W`.

The following questions ask for a matrix expression involving any of `A`, `Z`, `dLdA`, `dLdZ`, `W` and `W_0`.

**Enter your answers as Python expressions. You can use `transpose(x)` for transpose of an array, and `x@y` to indicate a matrix product of two arrays. Remember that `x*y` denotes component-wise multiplication.**

The `backward` method, given `dLdZ=\frac{\partial Loss}{\partial Z}` (an `n_l\times1` vector), returns `dLdA=\frac{\partial Loss}{\partial A}` (an `m_l\times1` vector):

$$
\frac{\partial Loss}{\partial A}
=
\frac{\partial Z}{\partial A}
\frac{\partial Loss}{\partial Z}
$$

<div align='center'>
    <img src='../assets/3_1_A.png' alt='3_1_A' style = 'border-radius: 10px'/>
</div>

The `backward` method, given `dLdZ=\frac{\partial Loss}{\partial Z}`, also computes `dLdW` (an $m_l\times n_l$ matrix) and `dLdW0` (an $n_l\times1$ vector), and stores them in the module instance.

<div align='center'>
    <img src='../assets/3_1_A.png' alt='3_1_A' style = 'border-radius: 10px'/>
</div>

<div align='center'>
    <img src='../assets/5_1_C.png' alt='5_1_C' style = 'border-radius: 10px'/>
</div>

We will use dLdW and dLdW0 as the gradient values in SGD.

### 1.2) Activation Module

Activation modules don't have any weights and so they are simpler.

The `forward` method for functions like *tanh* or *sigmoid*, given $Z$, return the function on the vector, componentwise. *Softmax* operates on the whole vector, as described earlier, and will need some special treatment.

The `backward` method, given **dLdA** $= \partial Loss / \partial A$, returns:

$$dLdZ = \frac{\partial Loss}{\partial Z} = \frac{\partial Loss}{\partial A} \frac{\partial A}{\partial Z}$$

In this case, $m^l = n^l$ and the quantities are column vectors of that size.

For $\text{Softmax} = SM(Z)$ at the output layer and assuming that we are using $NLL$ as the $Loss(A, Y)$ function, we have seen that there is a simple form for $\text{\textbf{dLdZ}} = \frac{\partial Loss}{\partial Z}$; namely, it is the prediction error $A - Y$. A similar result holds when using $NLL$ with a sigmoid output activation or a quadratic loss with a linear output activation. Note that for hinge loss with a linear activation, the form of `dLdZ` is different (see the lecture notes on the hinge loss).